In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("roydatascience/training-car")

print("Path to dataset files:", path)


Path to dataset files: /kaggle/input/training-car


In [2]:
IMG_path = '/kaggle/input/training-car/data/IMG/'
data_path = '/kaggle/input/training-car/data/driving_log.csv'

In [3]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import timm
import math
from pathlib import Path
from sklearn.model_selection import train_test_split
from typing import Tuple, List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# ========================= DATA PREPROCESSING =========================

class SteeringAngleAnalyzer:
    """Analyze and normalize steering angle distributions."""
    
    def __init__(self, steering_data: np.ndarray):
        self.steering_data = steering_data
        self.stats = {}
        
    def analyze_distribution(self) -> Dict:
        """Comprehensive steering angle distribution analysis."""
        data = self.steering_data
        
        self.stats = {
            'mean': np.mean(data),
            'std': np.std(data),
            'min': np.min(data),
            'max': np.max(data),
            'median': np.median(data),
            'q25': np.percentile(data, 25),
            'q75': np.percentile(data, 75),
            'zero_ratio': np.sum(data == 0) / len(data),
            'near_zero_ratio': np.sum(np.abs(data) < 0.01) / len(data),
            'range': np.max(data) - np.min(data)
        }
        
        return self.stats
    
    def print_stats(self):
        """Print comprehensive statistics."""
        if not self.stats:
            self.analyze_distribution()
            
        print("\n📊 STEERING ANGLE STATISTICS")
        print("="*40)
        print(f"Mean: {self.stats['mean']:.6f}")
        print(f"Std:  {self.stats['std']:.6f}")
        print(f"Min:  {self.stats['min']:.6f}")
        print(f"Max:  {self.stats['max']:.6f}")
        print(f"Range: {self.stats['range']:.6f}")
        print(f"Zero values: {self.stats['zero_ratio']:.2%}")
        print(f"Near-zero (±0.01): {self.stats['near_zero_ratio']:.2%}")
    
    def normalize_steering(self, method='minmax') -> np.ndarray:
        """Normalize steering angles using specified method."""
        data = self.steering_data
        
        if method == 'minmax':
            data_range = self.stats['max'] - self.stats['min']
            normalized = 2 * (data - self.stats['min']) / data_range - 1
        elif method == 'zscore':
            normalized = (data - self.stats['mean']) / self.stats['std']
        elif method == 'tanh':
            normalized = np.tanh(data / self.stats['std'])
        else:
            raise ValueError("Method must be 'minmax', 'zscore', or 'tanh'")
        
        print(f"✓ Normalized using {method}: range [{np.min(normalized):.3f}, {np.max(normalized):.3f}]")
        return normalized

class BalancedSampler:
    """Create balanced sampling weights for regression."""
    
    def __init__(self, steering_angles: np.ndarray, num_bins: int = 20):
        self.steering_angles = steering_angles
        self.num_bins = num_bins
        self.weights = None
        
    def create_binned_weights(self) -> np.ndarray:
        """Create sampling weights based on steering angle bins."""
        bin_edges = np.linspace(np.min(self.steering_angles), np.max(self.steering_angles), self.num_bins + 1)
        bin_indices = np.digitize(self.steering_angles, bin_edges) - 1
        bin_indices = np.clip(bin_indices, 0, self.num_bins - 1)
        
        bin_counts = np.bincount(bin_indices, minlength=self.num_bins)
        bin_weights = 1.0 / (bin_counts + 1e-8)
        sample_weights = bin_weights[bin_indices]
        sample_weights = sample_weights / np.sum(sample_weights) * len(sample_weights)
        
        self.weights = sample_weights
        return sample_weights
    
    def get_sampler(self) -> WeightedRandomSampler:
        """Get PyTorch WeightedRandomSampler."""
        if self.weights is None:
            self.create_binned_weights()
            
        return WeightedRandomSampler(
            weights=torch.FloatTensor(self.weights),
            num_samples=len(self.weights),
            replacement=True
        )

class MultiViewDataset(Dataset):
    """Dataset for multi-view car steering prediction."""
    
    def __init__(self, dataframe: pd.DataFrame, img_folder: str = 'IMG', 
                 transform: Optional[transforms.Compose] = None, steering_normalization: str = 'minmax'):
        self.df = dataframe.reset_index(drop=True)
        self.img_folder = Path(img_folder)
        self.transform = transform
        
        # Analyze and normalize steering angles
        self.analyzer = SteeringAngleAnalyzer(self.df.iloc[:, 3].values)  # 4th column is steering
        self.analyzer.analyze_distribution()
        self.normalized_steering = self.analyzer.normalize_steering(steering_normalization)
        
        # Create balanced sampler
        self.sampler = BalancedSampler(self.normalized_steering)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load three images
        center_path = self.img_folder / Path(row.iloc[0]).name
        left_path = self.img_folder / Path(row.iloc[1]).name
        right_path = self.img_folder / Path(row.iloc[2]).name
        
        # Load images with fallback
        try:
            center_img = Image.open(center_path).convert('RGB')
            left_img = Image.open(left_path).convert('RGB')
            right_img = Image.open(right_path).convert('RGB')
        except Exception:
            center_img = Image.new('RGB', (320, 160), color='black')
            left_img = Image.new('RGB', (320, 160), color='black')
            right_img = Image.new('RGB', (320, 160), color='black')
        
        # Apply transforms
        if self.transform:
            center_img = self.transform(center_img)
            left_img = self.transform(left_img)
            right_img = self.transform(right_img)
        
        # Get normalized steering angle
        steering = torch.FloatTensor([self.normalized_steering[idx]])
        
        return {
            'center': center_img,
            'left': left_img, 
            'right': right_img,
            'steering': steering
        }

class DataPreprocessor:
    """Main preprocessing pipeline."""
    
    def __init__(self, csv_path: str, img_folder: str):
        self.csv_path = csv_path
        self.img_folder = img_folder
        self.data = None
        self.train_dataset = None
        self.val_dataset = None
        
    def load_data(self):
        """Load the driving log CSV."""
        self.data = pd.read_csv(self.csv_path)
        print(f"✓ Loaded {len(self.data)} samples")
        return self.data
    
    def get_transforms(self, is_training: bool = True) -> transforms.Compose:
        """Get image transforms for training or validation."""
        
        if is_training:
            return transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                transforms.Lambda(lambda x: x + torch.randn_like(x) * 0.01)  # Gaussian noise
            ])
        else:
            return transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
    
    def create_datasets(self, steering_normalization: str = 'minmax', test_size: float = 0.2):
        """Create training and validation datasets."""
        if self.data is None:
            self.load_data()
        
        # Train/val split
        train_df, val_df = train_test_split(self.data, test_size=test_size, random_state=42, shuffle=True)
        print(f"✓ Train: {len(train_df)} samples, Val: {len(val_df)} samples")
        
        # Create datasets
        self.train_dataset = MultiViewDataset(train_df, self.img_folder, 
                                            self.get_transforms(True), steering_normalization)
        self.val_dataset = MultiViewDataset(val_df, self.img_folder, 
                                          self.get_transforms(False), steering_normalization)
        
        return self.train_dataset, self.val_dataset
    
    def create_dataloaders(self, batch_size: int = 32, use_balanced_sampling: bool = True, num_workers: int = 0):
        """Create training and validation dataloaders."""
        if self.train_dataset is None:
            self.create_datasets()
        
        # Get balanced sampler for training
        train_sampler = self.train_dataset.sampler.get_sampler() if use_balanced_sampling else None
        
        train_loader = DataLoader(self.train_dataset, batch_size=batch_size, sampler=train_sampler,
                                shuffle=(train_sampler is None), num_workers=num_workers, pin_memory=True)
        
        val_loader = DataLoader(self.val_dataset, batch_size=batch_size, shuffle=False, 
                              num_workers=num_workers, pin_memory=True)
        
        print(f"✓ Created dataloaders: batch_size={batch_size}, balanced_sampling={use_balanced_sampling}")
        return train_loader, val_loader

# ========================= MODEL ARCHITECTURE =========================

class MultiHeadCrossViewAttention(nn.Module):
    """Multi-head attention for fusing features from multiple camera views."""
    
    def __init__(self, embed_dim: int, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = 1.0 / math.sqrt(self.head_dim)
        
    def forward(self, center_feat: torch.Tensor, left_feat: torch.Tensor, right_feat: torch.Tensor) -> torch.Tensor:
        batch_size = center_feat.size(0)
        features = torch.stack([center_feat, left_feat, right_feat], dim=1)  # [batch_size, 3, embed_dim]
        
        # Project to Q, K, V
        Q = self.q_proj(features)
        K = self.k_proj(features)
        V = self.v_proj(features)
        
        # Reshape for multi-head attention
        Q = Q.view(batch_size, 3, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, 3, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, 3, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Apply attention
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, 3, self.embed_dim)
        
        # Output projection and global pooling
        output = self.out_proj(attn_output)
        fused_output = output.mean(dim=1)  # [batch_size, embed_dim]
        
        return fused_output

class BottleneckMLP(nn.Module):
    """MLP with bottleneck structure for regression."""
    
    def __init__(self, input_dim: int, hidden_dims: list = [512, 256, 128], output_dim: int = 1, dropout: float = 0.2):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
                nn.BatchNorm1d(hidden_dim)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        self.mlp = nn.Sequential(*layers)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(x)

class MultiViewSteeringModel(nn.Module):
    """Complete multi-view steering prediction model with attention fusion."""
    
    def __init__(self, backbone_name: str = 'resmlp_12_224', pretrained: bool = True,
                 num_attention_heads: int = 8, mlp_hidden_dims: list = [512, 256, 128],
                 dropout: float = 0.2, freeze_backbone: bool = False):
        super().__init__()
        
        # Load ResMLP backbone
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, 
                                        num_classes=0, global_pool='avg')
        self.feature_dim = self.backbone.num_features
        
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        # Selective fine-tuning: freeze first 60% of layers
        total_layers = len(list(self.backbone.named_parameters()))
        freeze_layers = int(total_layers * 0.6)
        
        print(f"🔒 Selective Fine-tuning: Freezing first {freeze_layers}/{total_layers} layers ({freeze_layers/total_layers:.1%})")
        
        frozen_count = 0
        trainable_count = 0
        
        for i, (name, param) in enumerate(self.backbone.named_parameters()):
            if i < freeze_layers:
                param.requires_grad = False
                frozen_count += 1
                print(f"  🔒 Frozen: {name}")
            else:
                trainable_count += 1
                print(f"  🔓 Trainable: {name}")
        
        print(f"✅ Backbone setup: {frozen_count} frozen, {trainable_count} trainable layers")
        
        # Multi-head attention for view fusion
        self.attention_fusion = MultiHeadCrossViewAttention(embed_dim=self.feature_dim,
                                                          num_heads=num_attention_heads, dropout=dropout)
        
        # Bottleneck MLP for regression
        self.regressor = BottleneckMLP(input_dim=self.feature_dim, hidden_dims=mlp_hidden_dims,
                                     output_dim=1, dropout=dropout)
        
        self._init_weights()
        
    def _init_weights(self):
        """Initialize model weights."""
        for module in [self.attention_fusion, self.regressor]:
            for m in module.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None:
                        nn.init.constant_(m.bias, 0)
                elif isinstance(m, nn.BatchNorm1d):
                    nn.init.constant_(m.weight, 1)
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, center: torch.Tensor, left: torch.Tensor, right: torch.Tensor) -> torch.Tensor:
        # Extract features from each view using shared backbone
        center_feat = self.backbone(center)
        left_feat = self.backbone(left)
        right_feat = self.backbone(right)
        
        # Fuse features using attention
        fused_feat = self.attention_fusion(center_feat, left_feat, right_feat)
        
        # Predict steering angle
        steering_pred = self.regressor(fused_feat)
        
        return steering_pred

class SteeringLoss(nn.Module):
    """Custom loss function for steering prediction."""
    
    def __init__(self, mse_weight: float = 1.0, smoothness_weight: float = 0.1):
        super().__init__()
        self.mse_weight = mse_weight
        self.smoothness_weight = smoothness_weight
        
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        mse_loss = F.mse_loss(pred, target)
        smoothness_loss = torch.mean(torch.abs(pred))
        total_loss = self.mse_weight * mse_loss + self.smoothness_weight * smoothness_loss
        return total_loss

class ModelTrainer:
    """Training pipeline for multi-view steering model."""
    
    def __init__(self, model: MultiViewSteeringModel, train_loader, val_loader, 
                 device: str = 'cuda', learning_rate: float = 1e-4, weight_decay: float = 1e-5):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        
        self.criterion = SteeringLoss()
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=100)
        
        self.train_losses = []
        self.val_losses = []
        
    def train_epoch(self) -> float:
        """Train for one epoch."""
        self.model.train()
        total_loss = 0.0
        num_batches = 0
        progress_bar = tqdm(self.train_loader, desc="Training")
        for batch in progress_bar:
            center = batch['center'].to(self.device)
            left = batch['left'].to(self.device)
            right = batch['right'].to(self.device)
            steering = batch['steering'].to(self.device)
            
            self.optimizer.zero_grad()
            pred_steering = self.model(center, left, right)
            loss = self.criterion(pred_steering, steering)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / num_batches
        self.train_losses.append(avg_loss)
        return avg_loss
    
    def validate(self) -> Tuple[float, float]:
        """Validate model."""
        self.model.eval()
        total_loss = 0.0
        total_mae = 0.0
        num_batches = 0
        
        with torch.no_grad():
            for batch in self.val_loader:
                center = batch['center'].to(self.device)
                left = batch['left'].to(self.device)
                right = batch['right'].to(self.device)
                steering = batch['steering'].to(self.device)
                
                pred_steering = self.model(center, left, right)
                loss = self.criterion(pred_steering, steering)
                mae = torch.mean(torch.abs(pred_steering - steering))
                
                total_loss += loss.item()
                total_mae += mae.item()
                num_batches += 1
        
        avg_loss = total_loss / num_batches
        avg_mae = total_mae / num_batches
        self.val_losses.append(avg_loss)
        return avg_loss, avg_mae
    
    def train(self, num_epochs: int = 50, print_freq: int = 5):
        """Full training loop."""
        print(f"🚀 Starting training for {num_epochs} epochs...")
        
        best_val_loss = float('inf')
        
        for epoch in range(num_epochs):
            train_loss = self.train_epoch()
            val_loss, val_mae = self.validate()
            self.scheduler.step()
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(self.model.state_dict(), 'best_steering_model.pth')
            
            if (epoch + 1) % print_freq == 0:
                print(f"Epoch {epoch+1}/{num_epochs}:")
                print(f"  Train Loss: {train_loss:.6f}")
                print(f"  Val Loss: {val_loss:.6f}")
                print(f"  Val MAE: {val_mae:.6f}")
                print(f"  LR: {self.optimizer.param_groups[0]['lr']:.2e}")
        
        print("✅ Training completed!")

# ========================= MAIN TRAINING SCRIPT =========================

def main():
    # Configuration
    CSV_PATH = data_path
    IMG_FOLDER = IMG_path
    BATCH_SIZE = 32
    NUM_EPOCHS = 50
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    print(f"Using device: {DEVICE}")
    
    # Step 1: Create preprocessed data
    print("📊 Setting up data preprocessing...")
    preprocessor = DataPreprocessor(CSV_PATH, IMG_FOLDER)
    train_dataset, val_dataset = preprocessor.create_datasets(steering_normalization='minmax')
    
    # Print data statistics
    train_dataset.analyzer.print_stats()
    
    # Step 2: Create data loaders
    train_loader, val_loader = preprocessor.create_dataloaders(
        batch_size=BATCH_SIZE, 
        use_balanced_sampling=True
    )
    
    # Step 3: Initialize model
    print("🏗️ Creating model...")
    model = MultiViewSteeringModel(
        backbone_name='resmlp_12_224',
        pretrained=True,
        num_attention_heads=8,
        mlp_hidden_dims=[512, 256, 128],
        dropout=0.2
    )
    
    print(f"📊 Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"📊 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    # Step 4: Initialize trainer and start training
    print("🎯 Starting training...")
    trainer = ModelTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        learning_rate=1e-4,
        weight_decay=1e-5
    )
    print("Testing data loading...")
    try:
        sample_batch = next(iter(train_loader))
        print(f"✓ Batch loaded: {sample_batch['center'].shape}")
    except Exception as e:
        print(f"❌ Data loading failed: {e}")
        return
    # Train the model
    trainer.train(num_epochs=NUM_EPOCHS, print_freq=5)
    
    print("🎉 Training pipeline completed!")

if __name__ == "__main__":
    main()

Using device: cuda
📊 Setting up data preprocessing...
✓ Loaded 8036 samples
✓ Train: 6428 samples, Val: 1608 samples
✓ Normalized using minmax: range [-1.000, 1.000]
✓ Normalized using minmax: range [-1.000, 1.000]

📊 STEERING ANGLE STATISTICS
Mean: 0.004862
Std:  0.129689
Min:  -0.942695
Max:  1.000000
Range: 1.942695
Zero values: 53.56%
Near-zero (±0.01): 54.78%
✓ Created dataloaders: batch_size=32, balanced_sampling=True
🏗️ Creating model...


model.safetensors:   0%|          | 0.00/61.4M [00:00<?, ?B/s]

🔒 Selective Fine-tuning: Freezing first 88/148 layers (59.5%)
  🔒 Frozen: stem.proj.weight
  🔒 Frozen: stem.proj.bias
  🔒 Frozen: blocks.0.ls1
  🔒 Frozen: blocks.0.ls2
  🔒 Frozen: blocks.0.norm1.alpha
  🔒 Frozen: blocks.0.norm1.beta
  🔒 Frozen: blocks.0.linear_tokens.weight
  🔒 Frozen: blocks.0.linear_tokens.bias
  🔒 Frozen: blocks.0.norm2.alpha
  🔒 Frozen: blocks.0.norm2.beta
  🔒 Frozen: blocks.0.mlp_channels.fc1.weight
  🔒 Frozen: blocks.0.mlp_channels.fc1.bias
  🔒 Frozen: blocks.0.mlp_channels.fc2.weight
  🔒 Frozen: blocks.0.mlp_channels.fc2.bias
  🔒 Frozen: blocks.1.ls1
  🔒 Frozen: blocks.1.ls2
  🔒 Frozen: blocks.1.norm1.alpha
  🔒 Frozen: blocks.1.norm1.beta
  🔒 Frozen: blocks.1.linear_tokens.weight
  🔒 Frozen: blocks.1.linear_tokens.bias
  🔒 Frozen: blocks.1.norm2.alpha
  🔒 Frozen: blocks.1.norm2.beta
  🔒 Frozen: blocks.1.mlp_channels.fc1.weight
  🔒 Frozen: blocks.1.mlp_channels.fc1.bias
  🔒 Frozen: blocks.1.mlp_channels.fc2.weight
  🔒 Frozen: blocks.1.mlp_channels.fc2.bias
  🔒 Fr

Training: 100%|██████████| 201/201 [03:13<00:00,  1.04it/s, loss=0.4027]


Epoch 5/50:
  Train Loss: 0.478521
  Val Loss: 0.110517
  Val MAE: 0.243113
  LR: 9.94e-05


...

Epoch 40/50:
  Train Loss: 0.067010
  Val Loss: 0.027499
  Val MAE: 0.100436
  LR: 6.55e-05


Training: 100%|██████████| 201/201 [03:06<00:00,  1.08it/s, loss=0.0927]


Epoch 45/50:
  Train Loss: 0.061669
  Val Loss: 0.026202
  Val MAE: 0.087775
  LR: 5.78e-05


Training: 100%|██████████| 201/201 [03:13<00:00,  1.04it/s, loss=0.0455]


Epoch 50/50:
  Train Loss: 0.059487
  Val Loss: 0.025830
  Val MAE: 0.095638
  LR: 5.00e-05
✅ Training completed!
🎉 Training pipeline completed!
